# Fase 2 — Ingesta Bronze

Excel (Volume) → Delta `ips_analytics.bronze.*` con metadatos `_ingested_at`, `_source_file`, `_batch_id`, `_row_hash`.

**Idempotencia (modo `full`):** cada corrida hace **overwrite** de las 4 tablas Bronze; no acumula duplicados.

Prerrequisitos: `00_create_catalog_schema.sql`, Excel en Volume, Repo enlazado o widget `repo_root`.

In [ ]:
%pip install openpyxl

In [ ]:
dbutils.widgets.text("repo_root", "", "Ruta Repo en Databricks (ej. /Workspace/Repos/user/ips-analytics)")
dbutils.widgets.text("raw_volume_path", "/Volumes/ips_analytics/raw/raw_data/", "Volume Excel")
dbutils.widgets.text("batch_id", "", "Batch ID (vacío = generar)")
dbutils.widgets.dropdown("load_mode", "full", ["full", "incremental"], "Modo de carga")

In [ ]:
import sys
from datetime import datetime, timezone
from uuid import uuid4

repo_root = dbutils.widgets.get("repo_root").rstrip("/")
if repo_root:
    src_path = f"{repo_root}/src"
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

from ips_analytics.config import DEFAULT_CONFIG, generate_batch_id
from ips_analytics.bronze.ingest_excel import log_pipeline_run, run_bronze_ingestion

raw_volume_path = dbutils.widgets.get("raw_volume_path")
load_mode = dbutils.widgets.get("load_mode")
batch_id = dbutils.widgets.get("batch_id").strip() or generate_batch_id()
run_id = str(uuid4())
started_at = datetime.now(timezone.utc)

print(f"batch_id={batch_id}")
print(f"raw_volume_path={raw_volume_path}")
print(f"load_mode={load_mode}")

In [ ]:
from pyspark.sql import functions as F

error_message = None
results = []
status = "success"
rows_in = rows_out = rows_rejected = 0

try:
    results = run_bronze_ingestion(
        spark,
        raw_volume_path=raw_volume_path,
        batch_id=batch_id,
        config=DEFAULT_CONFIG,
        load_mode=load_mode,
    )
    for r in results:
        print(
            f"{r.entity}: read={r.rows_read} bronze={r.rows_bronze} "
            f"quarantine={r.rows_quarantine} -> {r.table_fqn}"
        )
        rows_in += r.rows_read
        rows_out += r.rows_bronze
        rows_rejected += r.rows_quarantine
except Exception as exc:
    status = "failed"
    error_message = str(exc)
    raise
finally:
    ended_at = datetime.now(timezone.utc)
    try:
        log_pipeline_run(
            spark,
            run_id=run_id,
            batch_id=batch_id,
            stage="bronze",
            status=status,
            started_at=started_at,
            ended_at=ended_at,
            rows_in=rows_in,
            rows_out=rows_out,
            rows_rejected=rows_rejected,
            error_message=error_message,
        )
    except Exception as log_exc:
        print(f"WARN: no se pudo escribir ops.pipeline_runs: {log_exc}")

In [ ]:
catalog = DEFAULT_CONFIG.catalog
tables = ["pacientes", "citas", "eventos_clinicos", "facturacion"]

print("--- Conteos Bronze ---")
for t in tables:
    df = spark.table(f"{catalog}.bronze.{t}")
    n = df.count()
    batches = df.select("_batch_id").distinct().count()
    null_meta = df.filter(
        F.col("_batch_id").isNull()
        | F.col("_source_file").isNull()
        | F.col("_ingested_at").isNull()
    ).count()
    print(f"{t}: rows={n} distinct_batch={batches} null_metadata={null_meta}")
    assert null_meta == 0, f"Metadatos nulos en bronze.{t}"
    assert batches == 1, f"Se espera un solo batch_id activo (overwrite) en bronze.{t}"

print("\nValidación Bronze OK (idempotente: re-ejecutar overwrite reemplaza snapshot).")

In [ ]:
spark.table(f"{DEFAULT_CONFIG.catalog}.bronze.pacientes").select(
    "id_paciente", "_source_file", "_batch_id", "_ingested_at", "_row_hash"
).show(3, truncate=False)

## Cierre Fase 2

- Re-ejecutar este notebook: mismos conteos, un solo `_batch_id` por tabla.
- Siguiente paso: **Fase 3 — Silver** (`02_silver_transform`).